# OptiVision 0.6.1: real-hardware checks (Colab T4)

Run on a Colab T4 (Runtime, Change runtime type, T4 GPU), then Run all. It tests what the unit tests could not:
the released package with a real model, a folder that is not the sample, Qdrant export against a real server
and byaldi import against the real byaldi package. Each section prints PASS or FAIL and the last cell prints a
summary. A FAIL is a finding, not an error in the notebook. No Google Drive access is used.

In [ ]:
%pip install -q -U "optivision-rag[easy-st,qdrant,langchain,llamaindex,corpus]"
import optivision
print("optivision", optivision.__version__)

In [ ]:
import json, pathlib, shutil, traceback
import optivision.easy as ov

RESULTS = {}


def section(name):
    def deco(fn):
        try:
            detail = fn()
            RESULTS[name] = ("PASS", str(detail))
        except Exception as exc:
            RESULTS[name] = ("FAIL", type(exc).__name__ + ": " + str(exc)[:600])
            traceback.print_exc()
        print("==", name, RESULTS[name][0], "|", RESULTS[name][1])
        return fn
    return deco


MODEL = "vidore/colqwen2-v1.0-hf"
TARGET = 0.97

## A. The quickstart again on 0.6.1: does `check()` still call a fresh index 'needs attention'?

In [ ]:
@section("A quickstart + check() fix")
def _a():
    global idx_a
    folder = pathlib.Path("a_pdfs")
    folder.mkdir(exist_ok=True)
    shutil.copy(pathlib.Path(ov.__file__).parent / "data" / "sample.pdf", folder / "sample.pdf")
    idx_a = ov.build(str(folder), model=MODEL, target=TARGET, work_dir=None)
    c = idx_a.check()
    text = str(c)
    print(text)
    assert c.record.matches, "the build record still reports a difference"
    hits = idx_a.search("How do I get a golden yellow dye on wool?", k=3)
    assert hits[0].page == 20, "top hit is page " + str(hits[0].page)
    result_line = [ln for ln in text.splitlines() if ln.strip().startswith("Result")]
    return "record matches; top hit page 20; " + (result_line[0].strip() if result_line else "no Result line")

## B. A folder that is not the sample: two text PDFs and one scanned page (no text layer), then add and reload

In [ ]:
@section("B new folder: text and scan, search, report, add, reload")
def _b():
    global idx_b
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas
    from PIL import Image, ImageDraw, ImageFont

    d = pathlib.Path("b_pdfs")
    d.mkdir(exist_ok=True)

    def make(name, pages):
        c = canvas.Canvas(str(d / name), pagesize=A4)
        for title, body in pages:
            c.setFont("Helvetica-Bold", 20)
            c.drawString(60, 780, title)
            c.setFont("Helvetica", 12)
            y = 740
            for line in body:
                c.drawString(60, y, line)
                y -= 20
            c.showPage()
        c.save()

    make("finance.pdf", [
        ("Quarterly revenue by region", ["North: 4.2 million", "South: 3.1 million", "East: 5.7 million", "West: 2.9 million"]),
        ("Operating costs", ["Salaries are 62 percent of costs", "Cloud hosting is 14 percent", "Marketing is 9 percent"]),
        ("Risk register", ["Supplier concentration is the largest risk", "Currency exposure is hedged"]),
    ])
    make("garden.pdf", [
        ("Growing tomatoes", ["Plant after the last frost", "Water deeply twice a week", "Stake the vines early"]),
        ("Composting basics", ["Mix green kitchen scraps with brown leaves", "Turn the pile every two weeks"]),
    ])
    img = Image.new("RGB", (1240, 1754), "white")
    ImageDraw.Draw(img).text((120, 200), "NOTICE: The warehouse closes at 6 pm on Fridays.", fill="black",
                             font=ImageFont.load_default(size=44))
    img.save(d / "notice.pdf")

    idx_b = ov.build(str(d), model=MODEL, target=TARGET, work_dir=None)
    wants = {
        "Which region had the highest revenue?": "finance.pdf",
        "How often should I water tomato plants?": "garden.pdf",
        "When does the warehouse close on Friday?": "notice.pdf",
    }
    got = {}
    for q, f in wants.items():
        top = idx_b.search(q, k=1)[0]
        got[q] = top.id
        print(q, "->", top.id, round(top.score, 2))
    wrong = [q for q, f in wants.items() if f not in got[q]]
    assert not wrong, "wrong top hit for: " + str(wrong) + " got " + str(got)

    rep = idx_b.report()
    rep.save("b_report.html")
    rep.save("b_report.json")
    assert pathlib.Path("b_report.html").stat().st_size > 1000
    json.loads(pathlib.Path("b_report.json").read_text())

    make("travel.pdf", [("Museum hours", ["The city museum opens at 9 am and is closed on Mondays"])])
    sync = idx_b.add(str(d))
    print(sync)
    top = idx_b.search("Which day is the museum closed?", k=1)[0]
    assert "travel.pdf" in top.id, "added file not found: " + top.id

    before = [(h.id, round(h.score, 3)) for h in idx_b.search("Which region had the highest revenue?", k=3)]
    idx_b.close()
    idx_b = ov.load("b_pdfs.ovx")
    after = [(h.id, round(h.score, 3)) for h in idx_b.search("Which region had the highest revenue?", k=3)]
    assert before == after, str(before) + " != " + str(after)
    return "3/3 top hits right (incl. the scanned page); report html+json saved; added file found; reload gives the same hits"

## C. Qdrant export, local mode (real qdrant-client)

In [ ]:
@section("C Qdrant export (local folder)")
def _c():
    from qdrant_client import QdrantClient
    client = QdrantClient(path="qdrant_local")
    ex = idx_b.export_qdrant(client, "ov_local", recreate=True)
    n = client.count("ov_local").count
    assert n > 0
    return str(ex)[:300] + " | points=" + str(n)

## D. Qdrant export against a real Qdrant server (official release binary, started in this session)

In [ ]:
@section("D Qdrant export (real server)")
def _d():
    import subprocess, time, urllib.request
    subprocess.run("wget -q https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz -O q.tgz && tar xzf q.tgz",
                   shell=True, check=True)
    proc = subprocess.Popen(["./qdrant"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        for _ in range(40):
            try:
                urllib.request.urlopen("http://localhost:6333/readyz")
                break
            except Exception:
                time.sleep(1)
        ex = idx_b.export_qdrant("http://localhost:6333", "ov_server", recreate=True)
        from qdrant_client import QdrantClient
        client = QdrantClient("http://localhost:6333")
        n = client.count("ov_server").count
        assert n > 0
        return str(ex)[:300] + " | points=" + str(n)
    finally:
        proc.terminate()

## E. byaldi import against the real byaldi package (run last: installing byaldi may change library versions)

In [ ]:
%pip install -q byaldi

In [ ]:
@section("E byaldi import (real byaldi index)")
def _e():
    import subprocess
    subprocess.run("apt-get -qq install -y poppler-utils >/dev/null 2>&1", shell=True)
    from byaldi import RAGMultiModalModel
    m = RAGMultiModalModel.from_pretrained("vidore/colqwen2-v1.0", device="cuda")
    m.index(input_path="b_pdfs", index_name="b_byaldi", store_collection_with_index=False, overwrite=True)
    from optivision.easy.integrations.byaldi import import_index
    found = [p for p in pathlib.Path(".").rglob("b_byaldi") if p.is_dir()]
    assert found, "byaldi wrote no index folder"
    imported = import_index(found[0], out="b_from_byaldi.ovx")
    try:
        top = imported.search("Which region had the highest revenue?", k=1)[0]
        extra = " | search top: " + top.id
    except Exception as exc:
        extra = " | text search not run: " + type(exc).__name__ + ": " + str(exc)[:200]
    return "imported a real byaldi index" + extra

## Summary

In [ ]:
print(json.dumps(RESULTS, indent=2))
for name, (status, detail) in RESULTS.items():
    print(status, name)